In [25]:
!pip install groq pydantic llama-parse python-dotenv langchain langchain-community

In [26]:
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
print("API key loaded successfully")

API key loaded successfully


In [27]:
from pydantic import BaseModel
from typing import List, Optional, Any

# ── RESUME SCHEMA ──────────────────────────────────────────
class Experience(BaseModel):
    role: str
    company: str
    duration: str
    description: str

class Education(BaseModel):
    degree: str
    institution: str
    year: Optional[Any] = None

class ParsedResume(BaseModel):
    name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    skills: List[str]
    experience: List[Experience]
    education: List[Education]
    certifications: List[str] = []
    total_experience_years: Optional[float] = None

# ── JOB DESCRIPTION SCHEMA ─────────────────────────────────
class ParsedJD(BaseModel):
    job_title: str
    company: Optional[str] = None
    required_skills: List[str]
    preferred_skills: List[str] = []
    minimum_experience_years: Optional[float] = None
    education_requirement: Optional[str] = None
    certifications_required: List[str] = []
    responsibilities: List[str] = []
    location: Optional[str] = None

print("Resume schema created successfully")
print("Job Description schema created successfully")
print("Both schemas ready")

Resume schema created successfully
Job Description schema created successfully
Both schemas ready


In [28]:
import json
from groq import Groq

def parse_resume(resume_text: str) -> ParsedResume:

    client = Groq(api_key=api_key)

    prompt = f"""
    You are an expert resume parser for a production ATS system.
    Extract all information from the resume below.
    Return ONLY a valid JSON object, no markdown, no explanation.

    JSON must have exactly these fields:
    - name: string
    - email: string or null
    - phone: string or null
    - skills: list of strings (all technical and soft skills)
    - experience: list of objects with (role, company, duration, description)
    - education: list of objects with (degree, institution, year)
    - certifications: list of strings
    - total_experience_years: float (calculate total years of work experience)

    Resume:
    {resume_text}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a resume parser. Return valid JSON only. No markdown, no explanation."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    data = json.loads(raw)
    return ParsedResume(**data)

print("Resume parser function ready")

Resume parser function ready


In [29]:
def parse_jd(jd_text: str) -> ParsedJD:

    client = Groq(api_key=api_key)

    prompt = f"""
    You are an expert Job Description parser for a production ATS system.
    Extract all information from the job description below.
    Return ONLY a valid JSON object, no markdown, no explanation.

    JSON must have exactly these fields:
    - job_title: string
    - company: string or null
    - required_skills: list of strings (must-have skills)
    - preferred_skills: list of strings (nice-to-have skills)
    - minimum_experience_years: float or null
    - education_requirement: string or null
    - certifications_required: list of strings
    - responsibilities: list of strings (key job responsibilities)
    - location: string or null

    Job Description:
    {jd_text}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a JD parser. Return valid JSON only. No markdown, no explanation."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    data = json.loads(raw)
    return ParsedJD(**data)

print("JD parser function ready")

JD parser function ready


In [31]:
# Sample Resume
sample_resume = """
Charan Kumar
Email: charan@gmail.com
Phone: +91-9876543210

SKILLS
Python, Machine Learning, Deep Learning, NLP, TensorFlow, PyTorch,
SQL, Docker, Git, FastAPI, Scikit-learn, Pandas, NumPy

EXPERIENCE
Machine Learning Intern - TechStartup Pvt Ltd
June 2024 - December 2024
Built text classification models using BERT for customer feedback analysis.
Deployed ML models using FastAPI and Docker on AWS EC2.

Data Science Intern - Analytics Corp
January 2024 - May 2024
Performed exploratory data analysis on sales datasets using Pandas.
Created visualization dashboards using Matplotlib and Seaborn.

EDUCATION
Bachelor of Technology in Computer Science
IIIT Kota, 2025

CERTIFICATIONS
Deep Learning Specialization - Coursera (Andrew Ng)
AWS Cloud Practitioner
"""

# Sample Job Description
sample_jd = """
Company: Google India
Role: Machine Learning Engineer

Required Skills:
Python, Machine Learning, Deep Learning, NLP, TensorFlow or PyTorch, Docker, Git

Preferred Skills:
Kubernetes, MLflow, Hugging Face, LangChain, RAG systems

Minimum Experience: 1 year

Education Requirement:
Bachelor's degree in Computer Science or related field

Responsibilities:
- Build and deploy production ML models
- Design and implement NLP pipelines
- Monitor model performance in production

Location: Bangalore, India
"""

# Parse both
print("Parsing Resume...")
parsed_resume = parse_resume(sample_resume)
print("Resume parsed successfully")

print("Parsing Job Description...")
parsed_jd = parse_jd(sample_jd)
print("JD parsed successfully")

print("="*50)
print("PARSED RESUME:")
print("="*50)
print("Name:", parsed_resume.name)
print("Email:", parsed_resume.email)
print("Skills:", parsed_resume.skills)
print("Total Experience:", parsed_resume.total_experience_years, "years")
print("Education:", parsed_resume.education[0].degree, "from", parsed_resume.education[0].institution)
print("Certifications:", parsed_resume.certifications)

print()
print("="*50)
print("PARSED JOB DESCRIPTION:")
print("="*50)
print("Job Title:", parsed_jd.job_title)
print("Company:", parsed_jd.company)
print("Required Skills:", parsed_jd.required_skills)
print("Preferred Skills:", parsed_jd.preferred_skills)
print("Min Experience:", parsed_jd.minimum_experience_years, "years")
print("Location:", parsed_jd.location)

Parsing Resume...
Resume parsed successfully
Parsing Job Description...
JD parsed successfully
PARSED RESUME:
Name: Charan Kumar
Email: charan@gmail.com
Skills: ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'TensorFlow', 'PyTorch', 'SQL', 'Docker', 'Git', 'FastAPI', 'Scikit-learn', 'Pandas', 'NumPy']
Total Experience: 0.5 years
Education: Bachelor of Technology in Computer Science from IIIT Kota
Certifications: ['Deep Learning Specialization - Coursera (Andrew Ng)', 'AWS Cloud Practitioner']

PARSED JOB DESCRIPTION:
Job Title: Machine Learning Engineer
Company: Google India
Required Skills: ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'TensorFlow or PyTorch', 'Docker', 'Git']
Preferred Skills: ['Kubernetes', 'MLflow', 'Hugging Face', 'LangChain', 'RAG systems']
Min Experience: 1.0 years
Location: Bangalore, India


In [32]:
import json

# Save parsed resume to JSON
resume_dict = parsed_resume.model_dump()
with open("parsed_resume.json", "w") as f:
    json.dump(resume_dict, f, indent=2)

# Save parsed JD to JSON
jd_dict = parsed_jd.model_dump()
with open("parsed_jd.json", "w") as f:
    json.dump(jd_dict, f, indent=2)

print("parsed_resume.json saved successfully")
print("parsed_jd.json saved successfully")
print()
print("Resume JSON preview:")
print(json.dumps(resume_dict, indent=2))

parsed_resume.json saved successfully
parsed_jd.json saved successfully

Resume JSON preview:
{
  "name": "Charan Kumar",
  "email": "charan@gmail.com",
  "phone": "+91-9876543210",
  "skills": [
    "Python",
    "Machine Learning",
    "Deep Learning",
    "NLP",
    "TensorFlow",
    "PyTorch",
    "SQL",
    "Docker",
    "Git",
    "FastAPI",
    "Scikit-learn",
    "Pandas",
    "NumPy"
  ],
  "experience": [
    {
      "role": "Machine Learning Intern",
      "company": "TechStartup Pvt Ltd",
      "duration": "June 2024 - December 2024",
      "description": "Built text classification models using BERT for customer feedback analysis. Deployed ML models using FastAPI and Docker on AWS EC2."
    },
    {
      "role": "Data Science Intern",
      "company": "Analytics Corp",
      "duration": "January 2024 - May 2024",
      "description": "Performed exploratory data analysis on sales datasets using Pandas. Created visualization dashboards using Matplotlib and Seaborn."
    }
  